# Train RF-DETR Colony Detector on 22022540 Dataset (Kaggle)

This notebook trains an **RF-DETR Medium** detector on the **22022540 Colony Benchmark Dataset** (`anilanna4678/colony-dataset-22022540`), featuring 369 high-resolution agar plates across 24 microbial species and 56,865 annotations in COCO format.

### Settings:
- **Accelerator**: `GPU T4 x2` (or GPU P100)
- **Internet**: `On`
- **Dataset Input**: `anilanna4678/colony-dataset-22022540`

In [ ]:
# 1. GPU Check & Setup Workspace
import torch, os, sys, shutil, glob
assert torch.cuda.is_available(), 'GPU not found! Enable GPU T4 in Settings panel.'
print('GPU Device:', torch.cuda.get_device_name(0))
OUT = '/kaggle/working'

# Clone colony_detector repository
!rm -rf /kaggle/temp/colony_detector && git clone -q https://github.com/akashbabu9392/colony_detector.git /kaggle/temp/colony_detector
%cd /kaggle/temp/colony_detector
!pip -q install -r requirements.txt "rfdetr[train]>=1.7" pyyaml
print('Setup complete.')

In [ ]:
# 2. Locate 22022540 Dataset in /kaggle/input and Extract if Zipped
import glob, os, zipfile
from pathlib import Path

target_dir = Path('/kaggle/temp/dataset_22022540')
target_dir.mkdir(parents=True, exist_ok=True)

zip_candidates = glob.glob('/kaggle/input/**/*.zip', recursive=True)
main_zip = next((z for z in zip_candidates if 'dataset.zip' in z or '22022540' in z), None)

if main_zip and not (target_dir / 'annot_COCO.json').exists():
    print(f'Extracting {main_zip} to {target_dir}...')
    with zipfile.ZipFile(main_zip, 'r') as zf:
        zf.extractall(target_dir)
    data_dir = target_dir
else:
    coco_files = glob.glob('/kaggle/input/**/annot_COCO.json', recursive=True)
    if coco_files:
        data_dir = Path(coco_files[0]).parent
    else:
        data_dir = target_dir

jpgs = list(data_dir.rglob('*.jpg'))
cocos = list(data_dir.rglob('annot_COCO.json'))
print(f'Located dataset at: {data_dir}')
print(f'Images found: {len(jpgs)}, COCO annotations: {len(cocos)}')

In [ ]:
# 3. Slice Plates into Overlapping Native Tiles (build_training_set.py)
import sys, os

tile_out = '/kaggle/temp/tiles'
!rm -rf {tile_out}

# Slice the 369 plates into native 640x640 tiles with COCO annotations
!python tools/build_training_set.py \
    --dataset {data_dir} \
    --tile 640 \
    --overlap 0.25 \
    --coco \
    --out {tile_out} 2>&1 | tee /kaggle/working/tile_log.txt

print('Tiling finished. Tile directories:')
!ls -la {tile_out}

In [ ]:
# 4. Train RF-DETR Medium on the Tiles (10 Epochs)
import os, sys
OUT = '/kaggle/working'

!python tools/train_rfdetr.py \
    --dataset /kaggle/temp/tiles/coco \
    --size medium \
    --resolution 640 \
    --epochs 10 \
    --batch 4 \
    --grad-accum 4 \
    --lr 1e-4 \
    --output {OUT}/rfdetr \
    --out {OUT}/rfdetr_tiles.pth \
    2>&1 | grep -E --line-buffered "Epoch|Best|mAP|Error|Traceback|->" | tee {OUT}/train_log.txt

print('Training complete! Best EMA weights saved to', f'{OUT}/rfdetr_tiles.pth')

In [ ]:
# 5. Tune Thresholds on Validation Plates (tune_detector.py)
import os
OUT = '/kaggle/working'
os.environ['CD_RFDETR_WEIGHTS'] = f'{OUT}/rfdetr_tiles.pth'
os.environ['CD_ENGINES'] = 'rfdetr'

!python tools/tune_detector.py \
    --dataset {data_dir} \
    --engine rfdetr \
    --split valid \
    --conf-only \
    --cache /kaggle/temp/valid_dets.pkl | tee {OUT}/tune_log.txt

!cp models/detector.json {OUT}/detector.json
print('\nTuned detector configuration:')
!cat {OUT}/detector.json

print('\nFinal generated files in output:')
!ls -lh {OUT}